In [1]:
"""
FINAL ALIGNMENT SCRIPT: Hausa Mini-LLM GRPO
Optimized for: Tesla T4 (15GB VRAM)
Paper: "Efficient Alignment of Mini-LLMs for Hausa Language using GRPO and Unsloth"
"""

# ==============================================================================
# 0. INSTALL DEPENDENCIES (Required for Google Colab)
# ==============================================================================
import os

# Check if unsloth is installed; if not, install the necessary stack
try:
    import unsloth
except ImportError:
    print("Installing Unsloth and training dependencies...")
    # These specific versions and flags ensure compatibility with the Tesla T4
    os.system("pip install --quiet unsloth")
    os.system("pip install --quiet --no-deps trl peft accelerate bitsandbytes")
    os.system("pip install --quiet xformers")
    print("Installation complete. Proceeding to model initialization.")

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import re

# ==============================================================================
# 1. MODEL & PEFT INITIALIZATION (Section 2.1)
# ==============================================================================
model_name = "unsloth/Llama-3.2-3B-Instruct"
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False, # Stability for T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # From Appendix A
    lora_alpha = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"], # MLP included
    lora_dropout = 0,
    bias = "none",
)

# ==============================================================================
# 2. DATASET PREPARATION (The Alignment Corpus)
# ==============================================================================
hausa_data = [
    {"prompt": "Bayyana yadda ake yin tuwon shinkafa."},
    {"prompt": "Yaya ake girka miyar kuka dalla-dalla?"},
    {"prompt": "Yaya ya kamata a gaishe da dattijo a kasar Hausa?"},
    {"prompt": "Bani tarihin birnin Kano a takaice."},
    {"prompt": "Wace hanya ce mafi kyau ta wanke danyen kifi?"},
    {"prompt": "Wane ne sarkin Musulmi kuma ina fada take?"},
    {"prompt": "Idan bako ya zo gidanka, yaya kake karbarsa?"}
]
dataset = Dataset.from_list(hausa_data)

# ==============================================================================
# 3. REWARD FUNCTIONS (The GRPO Engine)
# ==============================================================================

def hausa_syntax_reward(prompts, completions, **kwargs):
    """
    Rewards completions using procedural Hausa markers (farko, sannan, wanke).
    Targets the 121% increase in syntactic coherence.
    """
    keywords = ["farko", "sannan", "kuma", "zuba", "wanke", "ruwa", "karshe"]
    rewards = []
    for completion in completions:
        # Check for density of Hausa function words
        count = sum(1 for word in keywords if word in completion.lower())
        rewards.append(1.0 if count >= 3 else 0.0)
    return rewards

def swahili_drift_penalty(prompts, completions, **kwargs):
    """
    Penalizes Swahili 'attractor' words (pika, katika, sana) identified in Table 2.
    Ensures cultural and linguistic anchoring.
    """
    swahili_indicators = ["pika", "katika", "habari", "sana", "nzuri", "kwanza"]
    rewards = []
    for completion in completions:
        found_drift = any(word in completion.lower() for word in swahili_indicators)
        rewards.append(-1.5 if found_drift else 0.5)
    return rewards

def repetition_penalty(prompts, completions, **kwargs):
    """
    Mitigates 'Linguistic Shadowing' (looping behavior).
    Calculates unique word ratio (U).
    """
    rewards = []
    for completion in completions:
        words = completion.split()
        if len(words) == 0:
            rewards.append(-1.0)
            continue
        u_ratio = len(set(words)) / len(words)
        # Threshold from paper: U >= 0.6 is good
        rewards.append(1.0 if u_ratio >= 0.6 else -1.0)
    return rewards

# ==============================================================================
# 4. TRAINING CONFIGURATION (Appendix A)
# ==============================================================================
training_args = GRPOConfig(
    output_dir = "hausa-llama-3b-grpo",
    learning_rate = 5e-6,      # From Table 2
    beta = 0.01,              # KL Coefficient
    num_generations = 8,       # Group Size G=8
    max_prompt_length = 256,
    max_completion_length = 512,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_steps = 20,            # Mini-run for validation
    logging_steps = 1,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    optim = "adamw_8bit",      # Memory efficiency for T4
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [hausa_syntax_reward, swahili_drift_penalty, repetition_penalty],
    args = training_args,
    train_dataset = dataset,
)

# ==============================================================================
# 5. EXECUTION & INFERENCE TEST
# ==============================================================================
print("\n[STEP 1] Starting GRPO Alignment...")
trainer.train()

print("\n[STEP 2] Testing Aligned Output (Anchoring Test)...")
FastLanguageModel.for_inference(model)

test_prompt = "Bayyana yadda ake yin tuwon shinkafa."
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    temperature = 0.7,
    pad_token_id = tokenizer.eos_token_id
)

print("\nALIGNED RESPONSE:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 6. Save final adapters
model.save_pretrained_merged("hausa_aligned_final", tokenizer, save_method = "lora")
print("\n✅ Alignment Complete. Model saved as 'hausa_aligned_final'")

Installing Unsloth and training dependencies...
Installation complete. Proceeding to model initialization.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:37<00:00, 48.84s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:34<00:00, 47.27s/it]


Unsloth: Merge process complete. Saved to `/content/hausa_aligned_final`

✅ Alignment Complete. Model saved as 'hausa_aligned_final'
